# Notebook 6 — Análisis Cualitativo Comparativo (SCE-Net · HGNN · Recommendation Engine)

Este notebook complementa los resultados cuantitativos ya reportados en el capítulo de Evaluación con un conjunto de análisis cualitativos, comunes a los tres modelos siempre que la arquitectura lo permita.

**Estructura:**
0. Setup y utilidades comunes
1. Carga de los tres modelos
2. Ejemplos cualitativos — Fill-in-the-Blank (FITB)
3. Ejemplos cualitativos — Compatibility Score
4. Calidad del espacio de embeddings (t-SNE/UMAP + Silhouette Score por categoría)
5. Coste computacional — nº de parámetros y tiempo de inferencia por tarea
6. Interpretación semántica de las máscaras de condición *(solo SCE-Net)*
7. Estructura de categorías aprendida *(solo HGNN)*
8. Errores sistemáticos de FITB por categoría *(los tres modelos)*
9. Triángulos rotos — evidencia empírica de no transitividad *(los tres modelos, conecta con RQ1 / §4.1.2)*


## 0. Setup y utilidades comunes

In [ ]:
import json, time, random, warnings
from pathlib import Path
from collections import defaultdict

import numpy as np
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
from PIL import Image
from sklearn.metrics import roc_auc_score, silhouette_score
from tqdm import tqdm

warnings.filterwarnings("ignore")

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
DATA_ROOT = Path("polyvore_kaggle/polyvore_outfits")   # ajustar si procede

# --- Estilo académico consistente con las figuras ya generadas para la memoria ---
plt.rcParams.update({
    "font.family": "serif", "font.size": 11, "axes.titlesize": 12,
    "axes.labelsize": 11, "legend.fontsize": 9.5,
    "xtick.labelsize": 10, "ytick.labelsize": 10,
    "axes.grid": True, "grid.alpha": 0.3, "grid.linestyle": "--",
    "axes.edgecolor": "0.3",
    "axes.prop_cycle": plt.cycler(color=plt.get_cmap("tab10").colors),
    "figure.dpi": 150,
})

MODEL_COLORS = {"SCE-Net": "tab:blue", "HGNN": "tab:orange", "RecEngine": "tab:green"}

print(f"Device: {DEVICE}")


In [ ]:
# --- Utilidades transversales a los tres modelos ---

def n_params(model) -> int:
    return sum(p.numel() for p in model.parameters())


def load_compatibility_file(txt_path, outfit_lookup, valid_ids=None):
    """Lee compatibility_test.txt ('label set_id_idx1 set_id_idx2 ...') y devuelve
    lista de (label:int, item_ids:list[str]). Igual formato para los 3 modelos --
    solo cambia qué outfit_lookup/valid_ids se le pasa."""
    samples = []
    with open(txt_path) as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) < 2:
                continue
            label = int(parts[0])
            item_ids = [outfit_lookup.get(k) for k in parts[1:]]
            item_ids = [i for i in item_ids if i is not None]
            if valid_ids is not None:
                item_ids = [i for i in item_ids if i in valid_ids]
            if len(item_ids) >= 2:
                samples.append((label, item_ids))
    return samples


def _load_img(root, item_index, item_id):
    meta = item_index.get(str(item_id), {})
    try:
        return Image.open(Path(root) / meta.get("image_path", "")).convert("RGB")
    except Exception:
        return Image.new("RGB", (224, 224), color="gray")


def plot_fitb_grid(root, item_index, query_ids, cand_ids, scores, gt_idx, title,
                    higher_is_better=True):
    """Misma disposición visual para los 3 modelos: fila 0 = outfit de consulta,
    fila 1 = candidatos (verde=GT, rojo=predicción incorrecta)."""
    predicted = int(np.argmax(scores)) if higher_is_better else int(np.argmin(scores))
    n_cols = max(len(query_ids), len(cand_ids))
    fig, axes = plt.subplots(2, n_cols, figsize=(2.6 * n_cols, 6))
    if n_cols == 1:
        axes = axes.reshape(2, 1)

    for i, item_id in enumerate(query_ids):
        meta = item_index.get(str(item_id), {})
        axes[0, i].imshow(_load_img(root, item_index, item_id))
        axes[0, i].set_title(meta.get("category", "?"), fontsize=9)
        axes[0, i].axis("off")
    for i in range(len(query_ids), n_cols):
        axes[0, i].axis("off")

    for i, (cand_id, score) in enumerate(zip(cand_ids, scores)):
        axes[1, i].imshow(_load_img(root, item_index, cand_id))
        color = "green" if i == gt_idx else ("red" if i == predicted else "black")
        label = f"s={score:.3f}"
        if i == gt_idx:
            label += " ✓GT"
        if i == predicted:
            label += " ←pred"
        axes[1, i].set_title(label, fontsize=8, color=color)
        axes[1, i].axis("off")
        for spine in axes[1, i].spines.values():
            spine.set_edgecolor(color); spine.set_linewidth(3); spine.set_visible(True)
    for i in range(len(cand_ids), n_cols):
        axes[1, i].axis("off")

    result = "✅ CORRECTO" if predicted == gt_idx else "❌ INCORRECTO"
    fig.suptitle(f"{title} — {result}", fontsize=12)
    plt.tight_layout()
    plt.show()


def pick_compat_examples(samples, score_fn, higher_is_better=True, max_samples=200, seed=0):
    """Puntúa una muestra de compatibility_test.txt y elige el ejemplo más 'confiado'
    de cada categoría de la matriz de confusión (TP/TN/FP/FN) según un umbral = mediana
    de los scores observados. Devuelve dict{'TP':(item_ids,score,label), ...} (las
    categorías sin ningún ejemplo en la muestra se omiten)."""
    rng = random.Random(seed)
    sample = samples if len(samples) <= max_samples else rng.sample(samples, max_samples)

    scored = []
    for label, item_ids in tqdm(sample, desc="Puntuando outfits"):
        s = score_fn(item_ids)
        if s is not None and np.isfinite(s):
            scored.append((label, item_ids, s))

    threshold = float(np.median([s for _, _, s in scored]))

    buckets = {"TP": [], "TN": [], "FP": [], "FN": []}
    for label, item_ids, s in scored:
        pred_compatible = (s > threshold) if higher_is_better else (s < threshold)
        actual_compatible = (label == 1)
        if pred_compatible and actual_compatible:
            key = "TP"
        elif not pred_compatible and not actual_compatible:
            key = "TN"
        elif pred_compatible and not actual_compatible:
            key = "FP"
        else:
            key = "FN"
        buckets[key].append((item_ids, s, label, abs(s - threshold)))

    chosen = {}
    for key, items in buckets.items():
        if items:
            items.sort(key=lambda x: -x[3])   # más confiado primero
            item_ids, s, label, _ = items[0]
            chosen[key] = (item_ids, s, label)
    return chosen, threshold


def plot_compat_outfit(root, item_index, item_ids, score, label, title):
    n = len(item_ids)
    fig, axes = plt.subplots(1, n, figsize=(2.6 * n, 3.2))
    if n == 1:
        axes = [axes]
    for ax, item_id in zip(axes, item_ids):
        meta = item_index.get(str(item_id), {})
        ax.imshow(_load_img(root, item_index, item_id))
        ax.set_title(meta.get("category", "?"), fontsize=8)
        ax.axis("off")
    lbl_txt = "compatible (GT)" if label == 1 else "incompatible (GT)"
    fig.suptitle(f"{title}\nscore={score:.3f} — {lbl_txt}", fontsize=11)
    plt.tight_layout()
    plt.show()


def time_inference(fn, *args, n_reps=30, n_warmup=5, **kwargs):
    """Media/desviación (ms) de n_reps llamadas a fn(*args, **kwargs), tras n_warmup
    pasadas de calentamiento. Sincroniza CUDA si procede, para medir tiempo real de GPU."""
    for _ in range(n_warmup):
        fn(*args, **kwargs)
    if DEVICE.type == "cuda":
        torch.cuda.synchronize()

    times = []
    for _ in range(n_reps):
        t0 = time.perf_counter()
        fn(*args, **kwargs)
        if DEVICE.type == "cuda":
            torch.cuda.synchronize()
        times.append((time.perf_counter() - t0) * 1000)
    return {"mean_ms": float(np.mean(times)), "std_ms": float(np.std(times)), "n_reps": n_reps}


## 1. Carga de los tres modelos

Cada modelo se carga en su propio *namespace* (prefijo `sce_`, `hgnn_`, `re_`) para evitar colisiones de nombres entre arquitecturas (las tres definen clases `Dataset`, funciones `evaluate_*`, etc. con nombres parecidos).

### 1.1 SCE-Net

In [ ]:
# ============================================================
# 1.1 SCE-Net -- calcado de 05_SCE-Net_evaluacion.ipynb (celdas 2, 4, 6-10)
# ============================================================
from torchvision import models as tv_models, transforms as tv_transforms

SCE_DATA_ROOT = DATA_ROOT
SCE_CKPT_PATH = Path("checkpoints/best_model_nondisjoint.pth")   # AJUSTAR si procede

# --- Arquitectura (idéntica a la del notebook 05) ---
class ResNet18Backbone(torch.nn.Module):
    def __init__(self, embed_dim):
        super().__init__()
        base = tv_models.resnet18(weights=tv_models.ResNet18_Weights.IMAGENET1K_V1)
        self.features = torch.nn.Sequential(*list(base.children())[:-1])
        self.projection = torch.nn.Linear(512, embed_dim, bias=False)

class ConditionMasks(torch.nn.Module):
    def __init__(self, m_conditions, embed_dim):
        super().__init__()
        self.masks = torch.nn.Parameter(torch.rand(m_conditions, embed_dim))

    def forward(self, v):
        return v.unsqueeze(1) * self.masks.unsqueeze(0)  # (B, M, D)

class ConditionWeightBranch(torch.nn.Module):
    def __init__(self, m_conditions, embed_dim, text_dim, hidden_dim=None, temperature=1.0):
        super().__init__()
        self.text_proj = torch.nn.Sequential(
            torch.nn.Linear(text_dim, 512), torch.nn.ReLU(), torch.nn.Dropout(0.3),
            torch.nn.Linear(512, embed_dim), torch.nn.ReLU(),
        )
        hidden = hidden_dim or max(2 * embed_dim, 64)
        self.mlp = torch.nn.Sequential(
            torch.nn.Linear(2 * embed_dim, hidden), torch.nn.ReLU(), torch.nn.Dropout(0.3),
            torch.nn.Linear(hidden, hidden),        torch.nn.ReLU(), torch.nn.Dropout(0.3),
            torch.nn.Linear(hidden, m_conditions),
        )
        self.temperature = temperature

    def forward(self, v_i, v_j, t_i, t_j):
        vt_i = v_i * self.text_proj(t_i)
        vt_j = v_j * self.text_proj(t_j)
        logits = self.mlp(torch.cat([vt_i, vt_j], dim=-1))
        return F.softmax(logits / self.temperature, dim=-1)

class SCENet(torch.nn.Module):
    def __init__(self, embed_dim, m_conditions, text_dim, img_size=224, temperature=1.0):
        super().__init__()
        self.backbone = ResNet18Backbone(embed_dim)
        self.cond_masks = ConditionMasks(m_conditions, embed_dim)
        self.weight_branch = ConditionWeightBranch(m_conditions, embed_dim, text_dim, temperature=temperature)

    def encode(self, img):
        feat = self.backbone.features(img).flatten(1)
        v_raw = self.backbone.projection(feat)
        v_norm = F.normalize(v_raw, p=2, dim=-1)
        return v_norm, v_raw

    def get_final_embedding(self, v_i, v_j, t_i, t_j):
        w = self.weight_branch(v_i, v_j, t_i, t_j)
        O_i = self.cond_masks(v_i)
        O_j = self.cond_masks(v_j)
        w_e = w.unsqueeze(-1)
        E_i = (w_e * O_i).sum(1)
        E_j = (w_e * O_j).sum(1)
        return F.normalize(E_i, p=2, dim=-1), F.normalize(E_j, p=2, dim=-1)

# --- Checkpoint ---
sce_ckpt = torch.load(SCE_CKPT_PATH, map_location=DEVICE, weights_only=False)
sce_config = sce_ckpt["config"]

sce_model = SCENet(
    embed_dim=sce_config["embed_dim"], m_conditions=sce_config["m_conditions"],
    text_dim=sce_config["text_dim"], img_size=sce_config["img_size"], temperature=0.1,
).to(DEVICE).eval()
sce_model.load_state_dict(sce_ckpt["model"])
print(f"[SCE-Net] cargado -- época {sce_ckpt['epoch']}, "
      f"valid_triplet {sce_ckpt.get('valid_triplet', sce_ckpt.get('valid_loss', 0.0)):.4f}")

# --- Recursos (lookup de items, Fisher Vectors, transform de imagen) ---
sce_outfit_lookup = {}
for split_file in ["nondisjoint/test.json", "nondisjoint/train.json", "nondisjoint/valid.json",
                    "disjoint/test.json", "disjoint/train.json", "disjoint/valid.json"]:
    with open(SCE_DATA_ROOT / split_file) as f:
        for outfit in json.load(f):
            set_id = outfit["set_id"]
            for item in outfit["items"]:
                sce_outfit_lookup[f"{set_id}_{item['index']}"] = str(item["item_id"])

with open(SCE_DATA_ROOT / "data/processed/item_index_full_nondisjoint.json") as f:
    sce_item_index = {str(k): v for k, v in json.load(f).items()}

with open(SCE_DATA_ROOT / "data/processed/fisher_item_ids_nondisjoint.json") as f:
    sce_fisher_ids = json.load(f)
sce_fisher_matrix = np.load(SCE_DATA_ROOT / "data/processed/fisher_vectors_nondisjoint.npy")
sce_fv_lookup = {str(item_id): sce_fisher_matrix[i] for i, item_id in enumerate(sce_fisher_ids)}
sce_zero_fv = np.zeros(sce_config["text_dim"], dtype=np.float32)

sce_transform = tv_transforms.Compose([
    tv_transforms.Resize(256), tv_transforms.CenterCrop(224), tv_transforms.ToTensor(),
    tv_transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

def sce_load_item_tensors(item_id: str):
    item_id = str(item_id)
    meta = sce_item_index.get(item_id)
    if meta is None:
        return None, None
    try:
        img = Image.open(SCE_DATA_ROOT / meta["image_path"]).convert("RGB")
        img_t = sce_transform(img).unsqueeze(0).to(DEVICE)
    except Exception:
        return None, None
    fv = sce_fv_lookup.get(item_id, sce_zero_fv)
    txt_t = torch.tensor(fv, dtype=torch.float32).unsqueeze(0).to(DEVICE)
    return img_t, txt_t

@torch.no_grad()
def sce_get_pair_distance(id_a: str, id_b: str):
    img_a, txt_a = sce_load_item_tensors(id_a)
    img_b, txt_b = sce_load_item_tensors(id_b)
    if img_a is None or img_b is None:
        return None
    v_a, _ = sce_model.encode(img_a)
    v_b, _ = sce_model.encode(img_b)
    E_a, E_b = sce_model.get_final_embedding(v_a, v_b, txt_a, txt_b)
    return (E_a - E_b).norm().item()

def sce_outfit_score(item_ids: list) -> float:
    """Distancia media entre todos los pares del outfit. Menor = más compatible."""
    ids = [str(i) for i in item_ids]
    distances = [sce_get_pair_distance(ids[i], ids[j])
                 for i in range(len(ids)) for j in range(i + 1, len(ids))]
    distances = [d for d in distances if d is not None]
    return float(np.mean(distances)) if distances else float("inf")

print(f"[SCE-Net] items indexados: {len(sce_item_index):,} | Fisher Vectors: {len(sce_fv_lookup):,}")


### 1.2 HGNN

In [ ]:
# ============================================================
# 1.2 HGNN -- calcado de 02_HGNN_entrenamiento.ipynb (celdas 2, 4, 6, 9-10, 12, 18-22)
#
# OJO: el checkpoint de HGNN es solo un state_dict (torch.save(model.state_dict(), ...)),
# no guarda el CONFIG ni category_to_idx -- así que para que los índices de categoría
# coincidan EXACTAMENTE con los usados en entrenamiento hace falta reconstruir el mismo
# pipeline de filtrado (mismo CONFIG, mismo split, mismo min_category_freq) antes de cargar
# los pesos. Si cambiaste CONFIG entre el entrenamiento real y esta celda, category_to_idx
# quedará desalineado con el Embedding aprendido -- usa el run_summary.json guardado en
# checkpoints_HGNN/ para verificar que el CONFIG de abajo coincide con el de esa ejecución.
# ============================================================
import scipy.sparse as sp

HGNN_POLYVORE_PATH = "./polyvore_kaggle/polyvore_outfits/"
HGNN_PATHS = {
    "bow_dir":       HGNN_POLYVORE_PATH + "data/preprocessed/BoW_for_HGNN/",
    "img_dir":       HGNN_POLYVORE_PATH + "data/preprocessed/InceptionV3_for_HGNN/",
    "text_features": HGNN_POLYVORE_PATH + "data/processed/text_features.csv",
    "checkpoints":   HGNN_POLYVORE_PATH + "checkpoints_HGNN/",
}
HGNN_CONFIG = {   # DEBE coincidir con el CONFIG usado en el entrenamiento real (ver run_summary.json)
    "split": "nondisjoint", "min_category_freq": 100, "min_outfit_items": 3,
    "hidden_dim": 16, "n_layers": 1, "beta": 0.2, "dropout_rate": 0.1,
}

# --- Splits y embeddings precomputados ---
hgnn_split_dir = HGNN_POLYVORE_PATH + HGNN_CONFIG["split"] + "/"
with open(hgnn_split_dir + "train.json") as f:
    hgnn_train_raw = json.load(f)
with open(hgnn_split_dir + "test.json") as f:
    hgnn_test_raw = json.load(f)

hgnn_img_embeddings = np.load(HGNN_PATHS["img_dir"] + "image_embeddings.npy")
hgnn_img_item_ids   = np.load(HGNN_PATHS["img_dir"] + "item_ids.npy", allow_pickle=True).astype(str)
hgnn_img_id_to_row  = {iid: i for i, iid in enumerate(hgnn_img_item_ids)}
HGNN_VIS_DIM = hgnn_img_embeddings.shape[1]

hgnn_bow_matrix   = sp.load_npz(HGNN_PATHS["bow_dir"] + "bow_features.npz")
hgnn_bow_item_ids = np.load(HGNN_PATHS["bow_dir"] + "item_ids.npy", allow_pickle=True).astype(str)
hgnn_bow_id_to_row = {iid: i for i, iid in enumerate(hgnn_bow_item_ids)}
HGNN_TXT_DIM = hgnn_bow_matrix.shape[1]

def hgnn_get_embeddings(item_id):
    ir, br = hgnn_img_id_to_row.get(item_id), hgnn_bow_id_to_row.get(item_id)
    if ir is None or br is None:
        return None
    vis = hgnn_img_embeddings[ir]
    txt = np.asarray(hgnn_bow_matrix[br].todense()).ravel().astype(np.float32)
    return vis, txt

# --- Categorías válidas (mismo filtro que en entrenamiento, sobre TRAIN) ---
df_text = pd.read_csv(HGNN_PATHS["text_features"], dtype={"item_id": str})
item_to_category = dict(zip(df_text["item_id"], df_text["sub_category"]))

from collections import Counter
cat_freq_train = Counter()
for outfit in hgnn_train_raw:
    for item in outfit["items"]:
        cat = item_to_category.get(str(item["item_id"]))
        if cat:
            cat_freq_train[cat] += 1

hgnn_valid_categories = sorted([c for c, n in cat_freq_train.items() if n >= HGNN_CONFIG["min_category_freq"]])
hgnn_category_to_idx  = {c: i for i, c in enumerate(hgnn_valid_categories)}
HGNN_N_CATEGORIES = len(hgnn_category_to_idx)
print(f"[HGNN] categorías válidas: {HGNN_N_CATEGORIES}")

def hgnn_filter_outfits(outfits_raw):
    filtered = []
    for outfit in outfits_raw:
        valid_items = []
        for item in outfit["items"]:
            item_id = str(item["item_id"])
            cat = item_to_category.get(item_id)
            if cat in hgnn_category_to_idx and hgnn_get_embeddings(item_id) is not None:
                valid_items.append({"item_id": item_id, "category": cat})
        if len(valid_items) >= HGNN_CONFIG["min_outfit_items"]:
            filtered.append({"items": valid_items})
    return filtered

hgnn_train_outfits = hgnn_filter_outfits(hgnn_train_raw)
hgnn_test_outfits  = hgnn_filter_outfits(hgnn_test_raw)
print(f"[HGNN] outfits test válidos: {len(hgnn_test_outfits)} / {len(hgnn_test_raw)}")

# --- Hipergrafo Theta (construido SOLO sobre train, igual que en entrenamiento) ---
def hgnn_build_hypergraph(outfits, category_to_idx):
    n_categories, n_outfits = len(category_to_idx), len(outfits)
    rows, cols, outfit_weights = [], [], []
    for outfit_idx, outfit in enumerate(outfits):
        outfit_categories = {item["category"] for item in outfit["items"]}
        for category in outfit_categories:
            rows.append(category_to_idx[category]); cols.append(outfit_idx)
        outfit_weights.append(len(outfit_categories))
    H = sp.csr_matrix((np.ones(len(rows)), (rows, cols)), shape=(n_categories, n_outfits))
    W = sp.diags(outfit_weights)
    D_v = np.array(H.sum(axis=1)).flatten()
    D_e = np.array(H.sum(axis=0)).flatten()
    return H, W, D_v, D_e

def hgnn_hypergraph_laplacian(H, W, D_v, D_e):
    D_v_inv_sqrt = sp.diags(np.where(D_v > 0, D_v ** -0.5, 0))
    D_e_inv = sp.diags(np.where(D_e > 0, D_e ** -1.0, 0))
    return D_v_inv_sqrt @ H @ W @ D_e_inv @ H.T @ D_v_inv_sqrt

_H, _W, _Dv, _De = hgnn_build_hypergraph(hgnn_train_outfits, hgnn_category_to_idx)
hgnn_Theta = torch.tensor(hgnn_hypergraph_laplacian(_H, _W, _Dv, _De).toarray(),
                           dtype=torch.float32, device=DEVICE)
print(f"[HGNN] Theta: {tuple(hgnn_Theta.shape)}")

# --- Arquitectura (idéntica a la del notebook de entrenamiento) ---
class HGNNPropagation(torch.nn.Module):
    def __init__(self, in_dim, hidden_dim, dropout_rate):
        super().__init__()
        self.linear  = torch.nn.Linear(in_dim, hidden_dim)
        self.norm    = torch.nn.LayerNorm(hidden_dim)
        self.dropout = torch.nn.Dropout(dropout_rate)

    def forward(self, X, Theta):
        X = torch.einsum("nm,bmd->bnd", Theta, X)   # propagacion sobre el hipergrafo categoria-categoria
        X = self.linear(X); X = self.norm(X); X = F.relu(X)
        return self.dropout(X)

class HGNNChannel(torch.nn.Module):
    """Copiado literal de 02_HGNN_entrenamiento.ipynb, celda 20 (forward completo)."""
    def __init__(self, input_dim, n_categories, hidden_dim, n_layers, dropout_rate):
        super().__init__()
        self.n_categories = n_categories
        node_dim = hidden_dim * 2
        self.input_norm = torch.nn.LayerNorm(input_dim)
        self.input_proj = torch.nn.Linear(input_dim, hidden_dim)
        self.category_embedding = torch.nn.Embedding(n_categories, hidden_dim)
        self.category_proj = torch.nn.Linear(hidden_dim, hidden_dim)
        self.prop_layers = torch.nn.ModuleList([
            HGNNPropagation(node_dim, node_dim, dropout_rate) for _ in range(n_layers)
        ])
        self.attn_importance = torch.nn.Linear(node_dim, 1)
        self.attn_compat     = torch.nn.Linear(node_dim, 1)
        self.repr_norm       = torch.nn.LayerNorm(node_dim)

    def forward(self, item_embeddings, cat_idx, mask, Theta):
        mask_exp = mask.unsqueeze(-1)

        normed = self.input_norm(item_embeddings)
        r_component = torch.tanh(self.input_proj(normed))
        c_i = self.category_embedding(cat_idx)
        c_component = torch.tanh(self.category_proj(c_i))
        r0 = torch.cat([r_component, c_component], dim=-1)
        r0 = r0 * mask_exp

        one_hot = F.one_hot(cat_idx, num_classes=self.n_categories).float() * mask_exp
        X0 = torch.einsum("bic,bid->bcd", one_hot, r0)

        X = X0
        for layer in self.prop_layers:
            X = layer(X, Theta)

        cat_idx_exp = cat_idx.unsqueeze(-1).expand(-1, -1, X.size(-1))
        item_repr = torch.gather(X, 1, cat_idx_exp)
        item_repr = self.repr_norm(item_repr)

        importance = torch.sigmoid(self.attn_importance(item_repr))
        compat = F.leaky_relu(self.attn_compat(item_repr), negative_slope=0.01)
        contrib = importance * compat * mask_exp
        score = contrib.squeeze(-1).sum(dim=1)

        return score, r0

class HGNNMultiModal(torch.nn.Module):
    def __init__(self, vis_dim, txt_dim, n_categories, hidden_dim, n_layers, beta, dropout_rate):
        super().__init__()
        self.beta = beta
        self.vis_channel = HGNNChannel(vis_dim, n_categories, hidden_dim, n_layers, dropout_rate)
        self.txt_channel = HGNNChannel(txt_dim, n_categories, hidden_dim, n_layers, dropout_rate)

    def forward(self, vis_emb, txt_emb, cat_idx, mask, Theta):
        score_vis, r_vis = self.vis_channel(vis_emb, cat_idx, mask, Theta)
        score_txt, r_txt = self.txt_channel(txt_emb, cat_idx, mask, Theta)
        score = self.beta * score_vis + (1.0 - self.beta) * score_txt
        mask_exp = mask.unsqueeze(-1)
        l_con = ((r_vis - r_txt) * mask_exp).pow(2).sum(dim=(1, 2))
        return score, l_con

hgnn_model = HGNNMultiModal(
    vis_dim=HGNN_VIS_DIM, txt_dim=HGNN_TXT_DIM, n_categories=HGNN_N_CATEGORIES,
    hidden_dim=HGNN_CONFIG["hidden_dim"], n_layers=HGNN_CONFIG["n_layers"],
    beta=HGNN_CONFIG["beta"], dropout_rate=HGNN_CONFIG["dropout_rate"],
).to(DEVICE)
hgnn_model.load_state_dict(torch.load(HGNN_PATHS["checkpoints"] + "best_model.pt", map_location=DEVICE))
hgnn_model.eval()
print(f"[HGNN] parámetros: {sum(p.numel() for p in hgnn_model.parameters()):,}")

# --- item_index equivalente al de SCE-Net, para poder pintar imágenes (HGNN no lo necesitaba
# en entrenamiento porque usa embeddings precomputados, no imágenes crudas) ---
hgnn_item_index = {
    iid: {"category": item_to_category.get(iid, "?"),
          "image_path": f"images/{iid}.jpg"}   # AJUSTAR si tu estructura de carpetas difiere
    for iid in set(hgnn_img_item_ids) | set(hgnn_bow_item_ids)
}

@torch.no_grad()
def hgnn_score_items(items: list, Theta=hgnn_Theta) -> float:
    """items: lista de dicts {'item_id','category'}. Idéntico a score_items del notebook 02."""
    hgnn_model.eval()
    n = len(items)
    vis = np.zeros((1, n, HGNN_VIS_DIM), dtype=np.float32)
    txt = np.zeros((1, n, HGNN_TXT_DIM), dtype=np.float32)
    cat = np.zeros((1, n), dtype=np.int64)
    mask = np.ones((1, n), dtype=np.float32)
    for i, item in enumerate(items):
        emb = hgnn_get_embeddings(item["item_id"])
        if emb is None:
            mask[0, i] = 0.0
            continue
        vis[0, i], txt[0, i] = emb
        cat[0, i] = hgnn_category_to_idx.get(item["category"], 0)
    score, _ = hgnn_model(
        torch.from_numpy(vis).to(DEVICE), torch.from_numpy(txt).to(DEVICE),
        torch.from_numpy(cat).to(DEVICE), torch.from_numpy(mask).to(DEVICE), Theta,
    )
    return score.item()


### 1.3 Recommendation Engine

In [ ]:
# ============================================================
# 1.3 Recommendation Engine -- calcado de Recommendation_Engine_v2_accelerated.ipynb
# (celdas 1, 3, 5, 7, 9, 11, 13, 20, 26)
# ============================================================
from sklearn.preprocessing import LabelEncoder

RE_POLYVORE_PATH = "./polyvore_kaggle/polyvore_outfits/"
RE_IMAGES_DIR    = RE_POLYVORE_PATH + "images/"
RE_OUTPUT_DIR     = RE_POLYVORE_PATH + "data/processed/"
RE_TEXT_EMBED_DIM = 100
RE_IMG_SIZE       = 224
RE_MAX_LEN        = 8

RE_CKPT_DIR      = RE_POLYVORE_PATH + "checkpoints_RE/"
RE_CKPT_PATH_CE   = RE_CKPT_DIR + "best_ce.pt"
RE_CKPT_PATH_FITB = RE_CKPT_DIR + "best_fitb.pt"
RE_CKPT_TO_USE    = RE_CKPT_PATH_FITB   # o RE_CKPT_PATH_CE -- decidir cuál usar por defecto

# --- Texto (GloVe precalculado) y metadatos por item ---
re_text_matrix = np.load(RE_OUTPUT_DIR + "glove_text_embeddings.npy")
with open(RE_OUTPUT_DIR + "glove_text_embeddings_index.json") as f:
    re_text_index = json.load(f)
re_item_id2textvec = {v: re_text_matrix[int(k)].astype(np.float32) for k, v in re_text_index.items()}

import pandas as pd
re_df = pd.read_csv(RE_OUTPUT_DIR + "glove_text_features.csv")
re_df["item_id"] = re_df["item_id"].astype(str)
re_cat_encoder = LabelEncoder()
re_df["cat_id"] = re_cat_encoder.fit_transform(re_df["sub_category"])
RE_NUM_CATEGORIES = re_df["cat_id"].nunique()

re_item_id2img_path = {iid: RE_IMAGES_DIR + f"{iid}.jpg" for iid in re_df["item_id"]}
re_item_id2cat       = dict(zip(re_df["item_id"], re_df["cat_id"]))

# item_index equivalente al de SCE-Net, para reutilizar las mismas funciones de visualización
# (O(n) con dict(zip(...)) -- la version anterior con .loc dentro del bucle era O(n^2) y por
# eso se quedaba "colgada" varios minutos con un catalogo de decenas de miles de items)
re_id2subcat = dict(zip(re_df["item_id"], re_df["sub_category"]))
re_item_index = {
    iid: {"category": cat, "image_path": f"images/{iid}.jpg"}
    for iid, cat in re_id2subcat.items()
}

# --- Outfit lookup (set_id_index -> item_id) ---
def re_build_outfit_lookup(json_paths):
    lookup = {}
    for path in json_paths:
        with open(path) as f:
            for outfit in json.load(f):
                for item in outfit["items"]:
                    lookup[f"{outfit['set_id']}_{item['index']}"] = str(item["item_id"])
    return lookup

re_outfit_lookup = re_build_outfit_lookup([
    RE_POLYVORE_PATH + "nondisjoint/train.json",
    RE_POLYVORE_PATH + "nondisjoint/valid.json",
    RE_POLYVORE_PATH + "nondisjoint/test.json",
])

re_transform = tv_transforms.Compose([
    tv_transforms.Resize(256), tv_transforms.CenterCrop(RE_IMG_SIZE), tv_transforms.ToTensor(),
    tv_transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

def re_load_item_tensors(item_id: str):
    item_id = str(item_id)
    path = re_item_id2img_path.get(item_id)
    if path is None:
        return None
    try:
        img = Image.open(path).convert("RGB")
    except Exception:
        return None
    img_t = re_transform(img).to(DEVICE)
    txt_t = torch.tensor(re_item_id2textvec.get(item_id, np.zeros(RE_TEXT_EMBED_DIM, dtype=np.float32)),
                          dtype=torch.float32, device=DEVICE)
    cat_t = torch.tensor(re_item_id2cat.get(item_id, 0), dtype=torch.long, device=DEVICE)
    return img_t, txt_t, cat_t

# --- Arquitectura (idéntica al notebook de entrenamiento) ---
class BasicBlock(torch.nn.Module):
    expansion = 1
    def __init__(self, in_planes, planes, stride=1, downsample=None):
        super().__init__()
        self.conv1 = torch.nn.Conv2d(in_planes, planes, 3, stride=stride, padding=1, bias=False)
        self.bn1 = torch.nn.BatchNorm2d(planes)
        self.relu = torch.nn.ReLU(inplace=True)
        self.conv2 = torch.nn.Conv2d(planes, planes, 3, stride=1, padding=1, bias=False)
        self.bn2 = torch.nn.BatchNorm2d(planes)
        self.downsample = downsample

    def forward(self, x):
        residual = x
        out = self.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        if self.downsample is not None:
            residual = self.downsample(x)
        return self.relu(out + residual)

def re_make_layer(in_planes, planes, num_blocks, stride=1):
    downsample = None
    if stride != 1 or in_planes != planes * BasicBlock.expansion:
        downsample = torch.nn.Sequential(
            torch.nn.Conv2d(in_planes, planes * BasicBlock.expansion, 1, stride=stride, bias=False),
            torch.nn.BatchNorm2d(planes * BasicBlock.expansion),
        )
    layers = [BasicBlock(in_planes, planes, stride=stride, downsample=downsample)]
    in_planes = planes * BasicBlock.expansion
    for _ in range(1, num_blocks):
        layers.append(BasicBlock(in_planes, planes))
    return torch.nn.Sequential(*layers), in_planes

class SmallResNet18(torch.nn.Module):
    def __init__(self, embedding_size=64):
        super().__init__()
        self.conv1 = torch.nn.Conv2d(3, 64, 7, stride=2, padding=3, bias=False)
        self.bn1 = torch.nn.BatchNorm2d(64)
        self.relu = torch.nn.ReLU(inplace=True)
        self.maxpool = torch.nn.MaxPool2d(3, stride=2, padding=1)
        in_planes = 64
        self.layer1, in_planes = re_make_layer(in_planes, 64, 2, stride=1)
        self.layer2, in_planes = re_make_layer(in_planes, 128, 2, stride=2)
        self.layer3, in_planes = re_make_layer(in_planes, 256, 2, stride=2)
        self.avgpool = torch.nn.AvgPool2d(kernel_size=7)
        self.flatten = torch.nn.Flatten()
        self.fc_embed = torch.nn.Linear(256 * 2 * 2, embedding_size)

    def forward(self, x):
        x = self.relu(self.bn1(self.conv1(x)))
        x = self.maxpool(x)
        x = self.layer1(x); x = self.layer2(x); x = self.layer3(x)
        x = self.avgpool(x); x = self.flatten(x)
        return self.fc_embed(x)

class TextEncoder(torch.nn.Module):
    def __init__(self, embedding_size=64, input_dim=RE_TEXT_EMBED_DIM):
        super().__init__()
        self.proj = torch.nn.Linear(input_dim, embedding_size)
        self.relu = torch.nn.ReLU(inplace=True)

    def forward(self, text_vec):
        return self.relu(self.proj(text_vec))

class CategoricalEncoder(torch.nn.Module):
    def __init__(self, num_categories, embedding_size=64):
        super().__init__()
        self.embedding = torch.nn.Embedding(num_categories, embedding_size)

    def forward(self, cat_ids):
        return self.embedding(cat_ids)

class ItemEncoder(torch.nn.Module):
    def __init__(self, num_categories, common_dim=128):
        super().__init__()
        self.image_encoder = SmallResNet18(embedding_size=common_dim)
        self.text_encoder = TextEncoder(embedding_size=common_dim)
        self.cat_encoder = CategoricalEncoder(num_categories, embedding_size=common_dim)
        self.norm = torch.nn.LayerNorm(common_dim, eps=1e-6)

    def forward(self, images, text_vecs, cat_ids):
        return self.norm(self.image_encoder(images) + self.text_encoder(text_vecs) + self.cat_encoder(cat_ids))

class TypeAwareGate(torch.nn.Module):
    def __init__(self, num_categories, d_model):
        super().__init__()
        self.type_embedding = torch.nn.Embedding(num_categories, d_model)
        self.gamma_proj = torch.nn.Linear(d_model, d_model)
        self.beta_proj = torch.nn.Linear(d_model, d_model)

    def forward(self, x, cat_ids):
        t = self.type_embedding(cat_ids)
        return x * torch.sigmoid(self.gamma_proj(t)) + self.beta_proj(t)

class TransformerEncoderBlock(torch.nn.Module):
    def __init__(self, d_model, n_heads, ffn_dim, dropout=0.1):
        super().__init__()
        self.attention = torch.nn.MultiheadAttention(d_model, n_heads, dropout=dropout, batch_first=True)
        self.ffn = torch.nn.Sequential(
            torch.nn.Linear(d_model, ffn_dim), torch.nn.ReLU(inplace=True), torch.nn.Dropout(dropout),
            torch.nn.Linear(ffn_dim, d_model),
        )
        self.norm1 = torch.nn.LayerNorm(d_model, eps=1e-6)
        self.norm2 = torch.nn.LayerNorm(d_model, eps=1e-6)
        self.drop1 = torch.nn.Dropout(dropout)
        self.drop2 = torch.nn.Dropout(dropout)

    def forward(self, x, key_padding_mask=None):
        attn_out, _ = self.attention(x, x, x, key_padding_mask=key_padding_mask, need_weights=False)
        x = self.norm1(x + self.drop1(attn_out))
        return self.norm2(x + self.drop2(self.ffn(x)))

class TAETransformerOutfitModel(torch.nn.Module):
    def __init__(self, num_categories, d_model=128, n_heads=4, ffn_dim=256, n_layers=2,
                 dropout=0.1, max_len=RE_MAX_LEN):
        super().__init__()
        self.max_len = max_len
        self.item_encoder = ItemEncoder(num_categories, common_dim=d_model)
        self.type_gate = TypeAwareGate(num_categories, d_model)
        self.cls_token = torch.nn.Parameter(torch.zeros(1, 1, d_model))
        self.encoder_layers = torch.nn.ModuleList([
            TransformerEncoderBlock(d_model, n_heads, ffn_dim, dropout) for _ in range(n_layers)
        ])
        self.dropout = torch.nn.Dropout(dropout)
        self.ce_head = torch.nn.Sequential(
            torch.nn.Linear(d_model, 64), torch.nn.ReLU(inplace=True), torch.nn.Dropout(dropout),
            torch.nn.Linear(64, 1), torch.nn.Sigmoid(),
        )
        self.fitb_head = torch.nn.Sequential(
            torch.nn.Linear(d_model * 2, 128), torch.nn.ReLU(inplace=True), torch.nn.Dropout(dropout),
            torch.nn.Linear(128, 64), torch.nn.ReLU(inplace=True), torch.nn.Dropout(dropout),
            torch.nn.Linear(64, 1),
        )

    def encode_items(self, images, text_vecs, cat_ids):
        B, L = images.shape[0], images.shape[1]
        item_emb = self.item_encoder(images.reshape(B * L, 3, RE_IMG_SIZE, RE_IMG_SIZE),
                                      text_vecs.reshape(B * L, RE_TEXT_EMBED_DIM), cat_ids.reshape(B * L))
        item_emb = self.type_gate(item_emb, cat_ids.reshape(B * L))
        return item_emb.reshape(B, L, -1)

    def encode_outfit(self, images, text_vecs, cat_ids, padding_mask=None):
        x = self.dropout(self.encode_items(images, text_vecs, cat_ids))
        batch_size = x.shape[0]
        cls = self.cls_token.expand(batch_size, -1, -1)
        x = torch.cat([cls, x], dim=1)
        key_padding_mask = None
        if padding_mask is not None:
            cls_mask = torch.zeros(batch_size, 1, dtype=torch.bool, device=x.device)
            key_padding_mask = torch.cat([cls_mask, padding_mask], dim=1)
        for layer in self.encoder_layers:
            x = layer(x, key_padding_mask=key_padding_mask)
        return F.normalize(x[:, 0, :], p=2, dim=-1)

    def call_ce(self, images, text_vecs, cat_ids, padding_mask=None):
        return self.ce_head(self.encode_outfit(images, text_vecs, cat_ids, padding_mask))

    def call_fitb(self, q_images, q_text, q_cats, cand_images, cand_text, cand_cats, q_mask=None):
        q_cls = self.encode_outfit(q_images, q_text, q_cats, q_mask)
        B, K = cand_images.shape[0], cand_images.shape[1]
        cand_emb = self.item_encoder(cand_images.reshape(B * K, cand_images.shape[2], RE_IMG_SIZE, RE_IMG_SIZE),
                                      cand_text.reshape(B * K, RE_TEXT_EMBED_DIM), cand_cats.reshape(B * K))
        cand_emb = F.normalize(self.type_gate(cand_emb, cand_cats.reshape(B * K)), p=2, dim=-1).reshape(B, K, -1)
        q_cls_expanded = q_cls.unsqueeze(1).expand(-1, K, -1)
        combined = torch.cat([q_cls_expanded, cand_emb], dim=-1).reshape(B * K, -1)
        return self.fitb_head(combined).reshape(B, K)

re_model = TAETransformerOutfitModel(num_categories=RE_NUM_CATEGORIES).to(DEVICE)
re_ckpt = torch.load(RE_CKPT_TO_USE, map_location=DEVICE)
re_model.load_state_dict(re_ckpt["model_state_dict"])   # sin optimizer -- no hace falta para inferencia
re_model.eval()
print(f"[RecEngine] cargado desde {RE_CKPT_TO_USE} -- "
      f"época {re_ckpt['epoch']}, best_metric {re_ckpt['best_metric']:.4f}")
print(f"[RecEngine] parámetros: {sum(p.numel() for p in re_model.parameters()):,}")

@torch.no_grad()
def re_score_ce(item_ids: list) -> float:
    """Compatibility score (probabilidad CE) de un outfit completo."""
    tensors = [re_load_item_tensors(i) for i in item_ids]
    tensors = [t for t in tensors if t is not None]
    if not tensors:
        return float("nan")
    L = len(tensors)
    images = torch.stack([t[0] for t in tensors]).unsqueeze(0)          # (1, L, 3, H, W)
    texts  = torch.stack([t[1] for t in tensors]).unsqueeze(0)          # (1, L, 100)
    cats   = torch.stack([t[2] for t in tensors]).unsqueeze(0)          # (1, L)
    mask   = torch.zeros(1, L, dtype=torch.bool, device=DEVICE)          # sin padding, todo válido
    return re_model.call_ce(images, texts, cats, mask).item()

@torch.no_grad()
def re_score_fitb(query_ids: list, candidate_ids: list) -> np.ndarray:
    """Devuelve un score por candidato (softmax opcional en la celda de visualización)."""
    q_tensors = [re_load_item_tensors(i) for i in query_ids]
    q_tensors = [t for t in q_tensors if t is not None]
    Lq = len(q_tensors)
    q_images = torch.stack([t[0] for t in q_tensors]).unsqueeze(0)
    q_texts  = torch.stack([t[1] for t in q_tensors]).unsqueeze(0)
    q_cats   = torch.stack([t[2] for t in q_tensors]).unsqueeze(0)
    q_mask   = torch.zeros(1, Lq, dtype=torch.bool, device=DEVICE)

    cand_tensors = [re_load_item_tensors(i) for i in candidate_ids]
    K = len(cand_tensors)
    cand_images = torch.stack([t[0] for t in cand_tensors]).unsqueeze(0)
    cand_texts  = torch.stack([t[1] for t in cand_tensors]).unsqueeze(0)
    cand_cats   = torch.stack([t[2] for t in cand_tensors]).unsqueeze(0)

    scores = re_model.call_fitb(q_images, q_texts, q_cats, cand_images, cand_texts, cand_cats, q_mask)
    return scores.squeeze(0).cpu().numpy()


## 2. Ejemplos cualitativos — Fill-in-the-Blank (FITB)

SCE-Net ya tiene esta visualización (notebook 05, celda 21). Aquí se replica para HGNN y RecEngine con la misma disposición visual (outfit de consulta arriba, candidatos abajo, GT en verde / predicción incorrecta en rojo), para que las tres figuras sean directamente comparables en la memoria.

In [ ]:
# ============================================================
# 2.1 SCE-Net -- FITB cualitativo (menor distancia = más compatible -> higher_is_better=False)
# ============================================================
with open(SCE_DATA_ROOT / "nondisjoint/fill_in_blank_test.json") as f:
    fitb_questions = json.load(f)   # compartido por los 3 modelos, mismas preguntas oficiales


def sce_visualize_fitb(question_idx: int):
    q = fitb_questions[question_idx]
    candidates = q["answers"]
    set_id = str(q["question"][0]).rsplit("_", 1)[0]
    gt_idx = next((i for i, c in enumerate(candidates) if c.rsplit("_", 1)[0] == set_id), 0)

    cand_ids  = [sce_outfit_lookup.get(c) for c in candidates]
    query_ids = [i for i in (sce_outfit_lookup.get(str(t)) for t in q["question"]) if i is not None]

    scores = []
    for cand_id in cand_ids:
        if cand_id is None:
            scores.append(float("inf")); continue
        dists = [sce_get_pair_distance(cand_id, qi) for qi in query_ids]
        dists = [d for d in dists if d is not None]
        scores.append(np.mean(dists) if dists else float("inf"))

    plot_fitb_grid(SCE_DATA_ROOT, sce_item_index, query_ids, cand_ids, scores, gt_idx,
                    title=f"SCE-Net — FITB pregunta {question_idx}", higher_is_better=False)


for idx in [0, 5, 10]:
    sce_visualize_fitb(idx)


In [ ]:
# ============================================================
# 2.2 HGNN -- FITB cualitativo (mayor score = más compatible -> higher_is_better=True)
# ============================================================
def hgnn_visualize_fitb(question_idx: int):
    q = fitb_questions[question_idx]
    candidates = q["answers"]
    set_id = str(q["question"][0]).rsplit("_", 1)[0]
    gt_idx = next((i for i, c in enumerate(candidates) if c.rsplit("_", 1)[0] == set_id), 0)

    cand_ids  = [sce_outfit_lookup.get(c) for c in candidates]     # mapeo set_id_idx->item_id es compartido
    query_ids = [i for i in (sce_outfit_lookup.get(str(t)) for t in q["question"]) if i is not None]

    context_items = [{"item_id": qi, "category": item_to_category.get(qi, "?")} for qi in query_ids]

    scores = []
    for cand_id in cand_ids:
        if cand_id is None or item_to_category.get(cand_id) is None:
            scores.append(float("-inf")); continue
        full_outfit = context_items + [{"item_id": cand_id, "category": item_to_category[cand_id]}]
        try:
            scores.append(hgnn_score_items(full_outfit))
        except Exception:
            scores.append(float("-inf"))

    plot_fitb_grid(HGNN_POLYVORE_PATH, hgnn_item_index, query_ids, cand_ids, scores, gt_idx,
                    title=f"HGNN — FITB pregunta {question_idx}", higher_is_better=True)


for idx in [0, 5, 10]:
    hgnn_visualize_fitb(idx)


In [ ]:
# ============================================================
# 2.3 RecEngine -- FITB cualitativo (usa call_fitb nativo, mayor score = más compatible)
# ============================================================
def re_visualize_fitb(question_idx: int):
    q = fitb_questions[question_idx]
    candidates = q["answers"]
    set_id = str(q["question"][0]).rsplit("_", 1)[0]
    gt_idx = next((i for i, c in enumerate(candidates) if c.rsplit("_", 1)[0] == set_id), 0)

    cand_ids  = [sce_outfit_lookup.get(c) for c in candidates]
    query_ids = [i for i in (sce_outfit_lookup.get(str(t)) for t in q["question"]) if i is not None]

    if any(c is None for c in cand_ids) or not query_ids:
        print(f"[RecEngine] pregunta {question_idx} saltada -- ids no resueltos")
        return

    raw_scores = re_score_fitb(query_ids, cand_ids)          # (K,) logits del fitb_head
    scores = F.softmax(torch.from_numpy(raw_scores), dim=0).numpy()   # solo para lectura, no cambia el argmax

    plot_fitb_grid(RE_POLYVORE_PATH, re_item_index, query_ids, cand_ids, scores.tolist(), gt_idx,
                    title=f"RecEngine — FITB pregunta {question_idx}", higher_is_better=True)


for idx in [0, 5, 10]:
    re_visualize_fitb(idx)


## 3. Ejemplos cualitativos — Compatibility Score

Análogo a la sección 2 pero para la tarea de Compatibility Prediction: se muestra un outfit completo (positivo o negativo, según `compatibility_test.txt`) junto con el score que le asigna el modelo, idealmente incluyendo casos de acierto y de error claro (falso positivo / falso negativo) para discusión.

In [ ]:
# ============================================================
# 3.1 SCE-Net -- Compatibility Score cualitativo (menor distancia = más compatible)
# ============================================================
sce_compat_samples = load_compatibility_file(
    SCE_DATA_ROOT / "nondisjoint/compatibility_test.txt", sce_outfit_lookup,
    valid_ids=set(sce_item_index.keys()),
)
print(f"[SCE-Net] muestras de compatibility_test.txt utilizables: {len(sce_compat_samples)}")

sce_compat_examples, sce_compat_threshold = pick_compat_examples(
    sce_compat_samples, sce_outfit_score, higher_is_better=False, max_samples=200,
)
print(f"[SCE-Net] umbral (mediana de distancias): {sce_compat_threshold:.3f}")
print(f"[SCE-Net] categorías encontradas: {list(sce_compat_examples.keys())}")

for key, (item_ids, score, label) in sce_compat_examples.items():
    plot_compat_outfit(SCE_DATA_ROOT, sce_item_index, item_ids, score, label,
                        title=f"SCE-Net — {key}")


In [ ]:
# ============================================================
# 3.2 HGNN -- Compatibility Score cualitativo (mayor score = más compatible)
# ============================================================
def hgnn_score_outfit_ids(item_ids: list) -> float:
    items = [{"item_id": i, "category": item_to_category.get(i)} for i in item_ids]
    items = [it for it in items if it["category"] is not None]
    if len(items) < 2:
        return None
    try:
        return hgnn_score_items(items)
    except Exception:
        return None

hgnn_compat_samples = load_compatibility_file(
    HGNN_POLYVORE_PATH + "nondisjoint/compatibility_test.txt", sce_outfit_lookup,
    valid_ids=set(hgnn_item_index.keys()),
)
print(f"[HGNN] muestras de compatibility_test.txt utilizables: {len(hgnn_compat_samples)}")

hgnn_compat_examples, hgnn_compat_threshold = pick_compat_examples(
    hgnn_compat_samples, hgnn_score_outfit_ids, higher_is_better=True, max_samples=200,
)
print(f"[HGNN] umbral (mediana de scores): {hgnn_compat_threshold:.3f}")
print(f"[HGNN] categorías encontradas: {list(hgnn_compat_examples.keys())}")

for key, (item_ids, score, label) in hgnn_compat_examples.items():
    plot_compat_outfit(HGNN_POLYVORE_PATH, hgnn_item_index, item_ids, score, label,
                        title=f"HGNN — {key}")


In [ ]:
# ============================================================
# 3.3 RecEngine -- Compatibility Score cualitativo (probabilidad CE, sigmoid en [0,1])
# ============================================================
re_compat_samples = load_compatibility_file(
    RE_POLYVORE_PATH + "nondisjoint/compatibility_test.txt", re_outfit_lookup,
    valid_ids=set(re_item_index.keys()),
)
print(f"[RecEngine] muestras de compatibility_test.txt utilizables: {len(re_compat_samples)}")

re_compat_examples, re_compat_threshold = pick_compat_examples(
    re_compat_samples, re_score_ce, higher_is_better=True, max_samples=200,
)
print(f"[RecEngine] umbral (mediana de probabilidades CE): {re_compat_threshold:.3f} "
      f"(nota: al ser sigmoid, 0.5 sería el umbral 'natural' -- se compara con la mediana "
      f"real observada para no asumir que el modelo está bien calibrado)")
print(f"[RecEngine] categorías encontradas: {list(re_compat_examples.keys())}")

for key, (item_ids, score, label) in re_compat_examples.items():
    plot_compat_outfit(RE_POLYVORE_PATH, re_item_index, item_ids, score, label,
                        title=f"RecEngine — {key}")


## 4. Calidad del espacio de embeddings

Para cada modelo: se extraen los embeddings finales de una muestra de ítems del test (misma muestra para los tres, para que sea comparable), se proyectan con t-SNE/UMAP coloreando por categoría, y se calcula el **Silhouette Score por categoría** (separabilidad real, no solo visual) como métrica cuantitativa de apoyo a la lectura cualitativa del embedding.

In [ ]:
# ============================================================
# 4.0 Utilidad común -- muestra compartida de items + calidad del embedding
# ============================================================
from sklearn.manifold import TSNE

try:
    import umap
    HAS_UMAP = True
except ImportError:
    HAS_UMAP = False
    print("umap-learn no está instalado -- se usará t-SNE (pip install umap-learn para tener ambos)")

# --- Muestra COMÚN a los 3 modelos: intersección de catálogos, para que la comparación
# sea justa (mismos items para los 3, no cada uno con su propio subconjunto disponible) ---
N_EMBED_SAMPLE = 300   # subir/bajar según el tiempo disponible -- cada item implica 3 forwards (1 por modelo)

common_ids = set(sce_item_index.keys()) & set(hgnn_item_index.keys()) & set(re_item_index.keys())
print(f"Items en la intersección de los 3 catálogos: {len(common_ids):,}")

_rng = random.Random(0)
embed_sample_ids = _rng.sample(sorted(common_ids), min(N_EMBED_SAMPLE, len(common_ids)))
embed_sample_categories = [item_to_category.get(i, sce_item_index.get(i, {}).get("category", "?"))
                            for i in embed_sample_ids]
print(f"Muestra usada para la sección 4: {len(embed_sample_ids)} items, "
      f"{len(set(embed_sample_categories))} categorías distintas")


def embedding_quality(embeddings: np.ndarray, categories: list, method: str = "tsne") -> dict:
    """Silhouette Score sobre el embedding ORIGINAL (no sobre la proyección 2D, que
    distorsiona distancias) + coordenadas 2D para la figura, con t-SNE o UMAP."""
    cats = np.asarray(categories)
    sil = silhouette_score(embeddings, cats) if len(set(categories)) > 1 else float("nan")

    if method == "umap" and HAS_UMAP:
        reducer = umap.UMAP(n_components=2, random_state=0)
    else:
        reducer = TSNE(n_components=2, random_state=0, init="pca",
                        perplexity=min(30, max(5, len(embeddings) // 4 - 1)))
    coords = reducer.fit_transform(embeddings)
    return {"coords": coords, "silhouette": float(sil), "categories": cats}


def plot_embedding_tsne(ax, result: dict, title: str, top_n_categories: int = 12):
    """Colorea solo las top_n_categories más frecuentes en la muestra (con tantas
    categorías como hay en Polyvore, un color por categoría deja de ser legible);
    el resto se pinta en gris como fondo."""
    cats = result["categories"]
    coords = result["coords"]
    top_cats = [c for c, _ in Counter(cats).most_common(top_n_categories)]
    cmap = plt.get_cmap("tab20")

    mask_other = ~np.isin(cats, top_cats)
    ax.scatter(coords[mask_other, 0], coords[mask_other, 1], color="0.85", s=10, label="_otras")
    for i, cat in enumerate(top_cats):
        mask = cats == cat
        ax.scatter(coords[mask, 0], coords[mask, 1], color=cmap(i % 20), s=14, label=cat)

    sil = result["silhouette"]
    ax.set_title(f"{title}\nSilhouette = {sil:.3f}", fontsize=10)
    ax.set_xticks([]); ax.set_yticks([])


In [ ]:
# ============================================================
# 4.1 SCE-Net -- embedding visual normalizado v_norm (antes de las máscaras de condición,
# que son PAIRWISE y no dan un vector fijo por item -- ver nota en la celda 4.0/markdown)
# ============================================================
@torch.no_grad()
def sce_item_embedding(item_id: str):
    img_t, _ = sce_load_item_tensors(item_id)
    if img_t is None:
        return None
    v_norm, _ = sce_model.encode(img_t)
    return v_norm.squeeze(0).cpu().numpy()

sce_embeddings, sce_embed_cats = [], []
for item_id, cat in tqdm(list(zip(embed_sample_ids, embed_sample_categories)), desc="SCE-Net embeddings"):
    emb = sce_item_embedding(item_id)
    if emb is not None:
        sce_embeddings.append(emb); sce_embed_cats.append(cat)
sce_embeddings = np.stack(sce_embeddings)

sce_embed_result = embedding_quality(sce_embeddings, sce_embed_cats, method="tsne")
print(f"[SCE-Net] Silhouette Score por categoría: {sce_embed_result['silhouette']:.4f}")


In [ ]:
# ============================================================
# 4.2 HGNN -- r0 del canal visual (concat de MLP(r_i) visual + MLP(c_i) categoría, ANTES
# de la propagación sobre el hipergrafo -- es la única representación realmente
# por-item de HGNNChannel; tras la propagación, item_repr queda agregado a nivel de
# categoría dentro del batch, no es específico del item -- ver forward() en la sección 1.2)
# ============================================================
@torch.no_grad()
def hgnn_item_embedding(item_id: str):
    emb = hgnn_get_embeddings(item_id)
    cat = item_to_category.get(item_id)
    if emb is None or cat not in hgnn_category_to_idx:
        return None
    vis_vec, _ = emb
    vis_t = torch.from_numpy(vis_vec).unsqueeze(0).to(DEVICE)
    cat_t = torch.tensor([hgnn_category_to_idx[cat]], dtype=torch.long, device=DEVICE)

    ch = hgnn_model.vis_channel
    r_component = torch.tanh(ch.input_proj(ch.input_norm(vis_t)))
    c_component = torch.tanh(ch.category_proj(ch.category_embedding(cat_t)))
    r0 = torch.cat([r_component, c_component], dim=-1)
    return r0.squeeze(0).cpu().numpy()

hgnn_embeddings, hgnn_embed_cats = [], []
for item_id, cat in tqdm(list(zip(embed_sample_ids, embed_sample_categories)), desc="HGNN embeddings"):
    emb = hgnn_item_embedding(item_id)
    if emb is not None:
        hgnn_embeddings.append(emb); hgnn_embed_cats.append(cat)
hgnn_embeddings = np.stack(hgnn_embeddings)

hgnn_embed_result = embedding_quality(hgnn_embeddings, hgnn_embed_cats, method="tsne")
print(f"[HGNN] Silhouette Score por categoría: {hgnn_embed_result['silhouette']:.4f}")


In [ ]:
# ============================================================
# 4.3 RecEngine -- salida de ItemEncoder + TypeAwareGate (fusión imagen+texto+categoría,
# ANTES del Transformer -- es la representación por-item más "rica" de los 3 porque ya
# combina las tres modalidades, a diferencia de SCE-Net/HGNN que aquí solo usan la
# modalidad visual -- tenlo en cuenta al comparar los 3 Silhouette Score, no es una
# comparación perfectamente simétrica en cuanto a qué información contiene cada vector)
# ============================================================
@torch.no_grad()
def re_item_embedding(item_id: str):
    tensors = re_load_item_tensors(item_id)
    if tensors is None:
        return None
    img_t, txt_t, cat_t = tensors
    item_emb = re_model.item_encoder(img_t.unsqueeze(0), txt_t.unsqueeze(0), cat_t.unsqueeze(0))
    item_emb = re_model.type_gate(item_emb, cat_t.unsqueeze(0))
    return item_emb.squeeze(0).cpu().numpy()

re_embeddings, re_embed_cats = [], []
for item_id, cat in tqdm(list(zip(embed_sample_ids, embed_sample_categories)), desc="RecEngine embeddings"):
    emb = re_item_embedding(item_id)
    if emb is not None:
        re_embeddings.append(emb); re_embed_cats.append(cat)
re_embeddings = np.stack(re_embeddings)

re_embed_result = embedding_quality(re_embeddings, re_embed_cats, method="tsne")
print(f"[RecEngine] Silhouette Score por categoría: {re_embed_result['silhouette']:.4f}")


In [ ]:
# ============================================================
# 4.4 Figura comparativa -- 1x3 t-SNE + tabla de Silhouette Score
# ============================================================
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
plot_embedding_tsne(axes[0], sce_embed_result, "SCE-Net")
plot_embedding_tsne(axes[1], hgnn_embed_result, "HGNN")
plot_embedding_tsne(axes[2], re_embed_result, "RecEngine")

handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, loc="lower center", ncol=6, fontsize=7, bbox_to_anchor=(0.5, -0.08))
fig.suptitle("Calidad del espacio de embeddings por categoría (t-SNE, misma muestra de items)", fontsize=13)
plt.tight_layout()
plt.savefig("/mnt/user-data/outputs/fig_embedding_quality.pdf", bbox_inches="tight")
plt.show()

print("\n=== Resumen -- Silhouette Score por categoría ===")
print(f"{'Modelo':<12} {'Silhouette':>10}")
for name, res in [("SCE-Net", sce_embed_result), ("HGNN", hgnn_embed_result), ("RecEngine", re_embed_result)]:
    print(f"{name:<12} {res['silhouette']:>10.4f}")


## 5. Coste computacional — parámetros y tiempo de inferencia

Nº de parámetros entrenables (HGNN ya lo imprime en su notebook de entrenamiento, celda 22; replicar igual para los otros dos) y tiempo de inferencia medido por tarea (FITB / Compatibility), con warm-up previo y varias repeticiones para reportar media ± desviación, mismo hardware/batch para los tres.

In [ ]:
# ============================================================
# 5.1 Número de parámetros entrenables
# ============================================================
params_table = {
    "SCE-Net":   n_params(sce_model),
    "HGNN":      n_params(hgnn_model),
    "RecEngine": n_params(re_model),
}
for name, n in params_table.items():
    print(f"{name:<12} {n:>12,} parámetros")


In [ ]:
# ============================================================
# 5.2 Tiempo de inferencia por tarea
#
# AVISO METODOLÓGICO IMPORTANTE: no es una comparación 100% simétrica en lo que cada
# modelo tiene que computar en inferencia --
#   - HGNN parte de embeddings visuales/textuales YA PRECOMPUTADOS (InceptionV3 + BoW,
#     calculados offline) -- su tiempo de inferencia NO incluye pasar la imagen por una CNN.
#   - SCE-Net y RecEngine sí decodifican la imagen y la pasan por su propio backbone
#     (ResNet18) en cada llamada.
# Esta asimetría es en sí misma un resultado relevante para la discusión (HGNN es más
# rápido en inferencia, pero a costa de depender de un pipeline de features offline que
# los otros dos no necesitan) -- no ocultarla, comentarla en la memoria.
# ============================================================

# --- Ejemplo fijo de FITB (mismas preguntas ya resueltas en la sección 2) ---
_fitb_q = fitb_questions[0]
_cand_ids  = [sce_outfit_lookup.get(c) for c in _fitb_q["answers"]]
_query_ids = [i for i in (sce_outfit_lookup.get(str(t)) for t in _fitb_q["question"]) if i is not None]

def _sce_fitb_call():
    for cid in _cand_ids:
        if cid is not None:
            for qid in _query_ids:
                sce_get_pair_distance(cid, qid)

def _hgnn_fitb_call():
    context_items = [{"item_id": qi, "category": item_to_category.get(qi, "?")} for qi in _query_ids]
    for cid in _cand_ids:
        if cid is not None and item_to_category.get(cid) is not None:
            hgnn_score_items(context_items + [{"item_id": cid, "category": item_to_category[cid]}])

def _re_fitb_call():
    re_score_fitb(_query_ids, [c for c in _cand_ids if c is not None])

# --- Ejemplo fijo de Compatibility (un outfit válido en la interseccion de los 3 catalogos) ---
_compat_item_ids = next(ids for _, ids in sce_compat_samples if set(ids).issubset(common_ids) and len(ids) >= 2)
print(f"Outfit usado para medir tiempo de Compatibility: {len(_compat_item_ids)} items")

def _sce_compat_call():
    sce_outfit_score(_compat_item_ids)

def _hgnn_compat_call():
    hgnn_score_outfit_ids(_compat_item_ids)

def _re_compat_call():
    re_score_ce(_compat_item_ids)

timing_table = {}
for name, fitb_fn, compat_fn in [
    ("SCE-Net",   _sce_fitb_call,  _sce_compat_call),
    ("HGNN",      _hgnn_fitb_call, _hgnn_compat_call),
    ("RecEngine", _re_fitb_call,   _re_compat_call),
]:
    t_fitb   = time_inference(fitb_fn,  n_reps=30, n_warmup=5)
    t_compat = time_inference(compat_fn, n_reps=30, n_warmup=5)
    timing_table[name] = {"fitb": t_fitb, "compat": t_compat}
    print(f"{name:<12} FITB: {t_fitb['mean_ms']:6.2f} ± {t_fitb['std_ms']:5.2f} ms   "
          f"Compat: {t_compat['mean_ms']:6.2f} ± {t_compat['std_ms']:5.2f} ms")


In [ ]:
# ============================================================
# 5.3 Tabla resumen (LaTeX) -- parámetros + tiempo de inferencia por tarea
# ============================================================
latex_lines = []
latex_lines.append(r"\begin{table}[htbp]")
latex_lines.append(r"    \centering")
latex_lines.append(r"    \begin{tabular}{l r r r}")
latex_lines.append(r"        \toprule")
latex_lines.append(r"        Modelo & Parámetros & Tiempo FITB (ms) & Tiempo Compat. (ms) \\")
latex_lines.append(r"        \midrule")
for name in ["SCE-Net", "HGNN", "RecEngine"]:
    p = params_table[name]
    t_fitb = timing_table[name]["fitb"]
    t_compat = timing_table[name]["compat"]
    latex_lines.append(
        f"        {name} & {p:,} & "
        f"{t_fitb['mean_ms']:.2f} $\\pm$ {t_fitb['std_ms']:.2f} & "
        f"{t_compat['mean_ms']:.2f} $\\pm$ {t_compat['std_ms']:.2f} \\\\"
    )
latex_lines.append(r"        \bottomrule")
latex_lines.append(r"    \end{tabular}")
latex_lines.append(
    r"    \caption{Número de parámetros entrenables y tiempo de inferencia por tarea "
    r"(media $\pm$ desviación típica sobre 30 repeticiones, tras 5 de calentamiento). "
    r"HGNN parte de embeddings visuales/textuales precomputados offline, mientras que "
    r"SCE-Net y Recommendation Engine procesan la imagen cruda en cada inferencia -- "
    r"los tiempos no son directamente comparables sin tener esto en cuenta (ver discusión).}"
)
latex_lines.append(r"    \label{tab:coste_computacional}")
latex_lines.append(r"\end{table}")

latex_table = "\n".join(latex_lines)
print(latex_table)

with open("/mnt/user-data/outputs/tabla_coste_computacional.tex", "w") as f:
    f.write(latex_table)


## 6. Interpretación semántica de las máscaras de condición *(solo SCE-Net)*

Ya implementado en el notebook 05 (celda 23) -- se incorpora aquí tal cual para que la memoria tenga todo el análisis cualitativo reunido en un único notebook de referencia.

In [ ]:
# ============================================================
# 6.1 Interpretación semántica de las máscaras de condición (SCE-Net)
# Calcado literal de 05_SCE-Net_evaluacion.ipynb, celda 23
# ============================================================
@torch.no_grad()
def sce_get_condition_weights(id_a: str, id_b: str):
    img_a, txt_a = sce_load_item_tensors(id_a)
    img_b, txt_b = sce_load_item_tensors(id_b)
    if img_a is None or img_b is None:
        return None
    v_a, _ = sce_model.encode(img_a)
    v_b, _ = sce_model.encode(img_b)
    w = sce_model.weight_branch(v_a, v_b, txt_a, txt_b)
    return w.squeeze(0).cpu().numpy()

cat_weights = {}   # (cat_a, cat_b) -> lista de vectores w
for q in tqdm(fitb_questions[:200], desc="Analizando pesos de condición"):
    ids = [i for i in (sce_outfit_lookup.get(str(t)) for t in q["question"]) if i is not None]
    for i in range(len(ids)):
        for j in range(i + 1, len(ids)):
            w = sce_get_condition_weights(ids[i], ids[j])
            if w is None:
                continue
            cat_a = sce_item_index.get(ids[i], {}).get("category", "?")
            cat_b = sce_item_index.get(ids[j], {}).get("category", "?")
            key = tuple(sorted([cat_a, cat_b]))
            cat_weights.setdefault(key, []).append(w)

top_pairs = sorted(cat_weights.items(), key=lambda x: -len(x[1]))[:8]
M = sce_model.weight_branch.mlp[-1].out_features

fig, axes = plt.subplots(2, 4, figsize=(14, 6))
axes = axes.flatten()
for ax, (pair, ws) in zip(axes, top_pairs):
    mean_w = np.mean(ws, axis=0)
    ax.bar(range(1, M + 1), mean_w, color="steelblue")
    ax.set_title(f"{pair[0]}\n\u2194 {pair[1]}\n(n={len(ws)})", fontsize=8)
    ax.set_xlabel("Condición", fontsize=7)
    ax.set_ylabel("Peso medio", fontsize=7)
    ax.set_ylim(0, 1)
    ax.set_xticks(range(1, M + 1))

plt.suptitle("SCE-Net — Distribución media de pesos de condición por tipo de par", fontsize=11)
plt.tight_layout()
plt.show()


#### 6.1-DIAG — Diagnóstico: ¿colapso real del *weight branch* o problema de datos/escala?

Antes de interpretar la figura anterior, comprobar: (1) los valores exactos por panel (no leídos a ojo sobre una barra con eje fijo 0-1), (2) si las medias por tipo de par coinciden con la media global agrupando todos los pares (lo que confirmaría que el hallazgo es real y no un artefacto de agregación), y (3)-(4) si los propios inputs (Fisher Vectors y embeddings visuales) varían de verdad entre items distintos.

In [ ]:
# ============================================================
# 6.1-DIAG -- Diagnóstico: ¿colapso real del modelo o problema de datos/escala?
# ============================================================

# 1) Valores EXACTOS por panel (no leídos de la barra a ojo)
print("=== Medias exactas por par de categorías ===")
for pair, ws in top_pairs:
    mean_w = np.mean(ws, axis=0)
    std_w = np.std(ws, axis=0)
    print(f"{pair[0]:>12} <-> {pair[1]:<12} (n={len(ws):3d})  mean={np.round(mean_w, 4)}  std={np.round(std_w, 3)}")

# 2) Media GLOBAL (pooled, ignorando el tipo de par) para comparar
all_ws = [w for ws in cat_weights.values() for w in ws]
print(f"\nMedia GLOBAL (todos los pares mezclados, n={len(all_ws)}): {np.round(np.mean(all_ws, axis=0), 4)}")
print("-> Si las medias 'por par' de arriba son básicamente esta media global, el hallazgo "
      "es real (el weight_branch no distingue tipo de par) y no un bug de agregación.")

# 3) ¿Varían de verdad los inputs? Si t_i (Fisher Vector) es casi constante entre items
# distintos, el problema está en el lookup de texto (celda 6), no en el modelo ni en la 30.
sample_ids = list(sce_item_index.keys())[:50]
text_vecs = []
for iid in sample_ids:
    fv = sce_fv_lookup.get(iid)
    if fv is not None:
        text_vecs.append(fv)
text_vecs = np.stack(text_vecs) if text_vecs else np.zeros((0, sce_config["text_dim"]))
n_zero = sum(1 for v in text_vecs if np.allclose(v, 0))
print(f"\nDe {len(sample_ids)} items de muestra: {len(text_vecs)} tienen Fisher Vector encontrado, "
      f"{n_zero} de ellos son exactamente el vector cero (fallback).")
print(f"Std medio de los Fisher Vectors encontrados (por dimensión, promediado): "
      f"{text_vecs.std(axis=0).mean():.4f}  (si esto es ~0, todos los items tienen el mismo vector de texto)")

# 4) ¿Varía v_i (visual) entre items? Si esto también es ~constante, el problema es de
# carga de imagen, no del weight_branch.
img_embs = []
for iid in sample_ids[:20]:
    img_t, _ = sce_load_item_tensors(iid)
    if img_t is not None:
        v, _ = sce_model.encode(img_t)
        img_embs.append(v.squeeze(0).cpu().numpy())
img_embs = np.stack(img_embs) if img_embs else np.zeros((0, sce_config["embed_dim"]))
print(f"\nStd medio de los embeddings visuales v_norm (por dimensión, promediado): "
      f"{img_embs.std(axis=0).mean():.4f}  (si esto es razonablemente >0, la imagen sí varía por item)")


## 7. Estructura de categorías aprendida *(solo HGNN)*

`HGNNChannel.category_embedding` es un `nn.Embedding(n_categories=158, hidden_dim)` — un vector aprendido por categoría, independiente del ítem concreto (Ec. 1 del paper de Feng et al.). Se proyecta con t-SNE/UMAP y se colorea por macro-categoría (superior/inferior/calzado/accesorio, etc.) para comprobar si categorías semánticamente próximas (camisetas/blusas) quedan próximas en el espacio aprendido -- evidencia visual adicional para la discusión ya existente sobre el patrón FITB/AUC de HGNN.

In [ ]:
# ============================================================
# 7.1 Extraer los embeddings de categoría aprendidos (HGNN)
# ============================================================
hgnn_cat_emb_vis = hgnn_model.vis_channel.category_embedding.weight.detach().cpu().numpy()   # (158, hidden_dim)
hgnn_cat_emb_txt = hgnn_model.txt_channel.category_embedding.weight.detach().cpu().numpy()

hgnn_idx_to_category = {v: k for k, v in hgnn_category_to_idx.items()}
hgnn_category_names = [hgnn_idx_to_category[i] for i in range(HGNN_N_CATEGORIES)]

print(f"category_embedding (canal visual):  {hgnn_cat_emb_vis.shape}")
print(f"category_embedding (canal textual): {hgnn_cat_emb_txt.shape}")
print(f"Ejemplo de categorías: {hgnn_category_names[:10]}")


In [ ]:
# ============================================================
# 7.2 t-SNE/UMAP de las 158 categorías, coloreado por macro-categoría
#
# AVISO: no existe un mapeo categoría->macro-categoría en tus notebooks de origen, así
# que aquí uso un clasificador heurístico por palabras clave sobre el NOMBRE de la
# categoría (sub_category de Polyvore). Es una heurística razonable para una primera
# lectura visual, pero REVISA hgnn_category_names impreso arriba y ajusta las listas
# de palabras clave de abajo si alguna categoría de tu dataset queda mal clasificada
# como "otros" o en el grupo equivocado -- no la des por buena sin mirarla.
# ============================================================
MACRO_CATEGORY_KEYWORDS = {
    "tops":       ["shirt", "blouse", "top", "tee", "tank", "sweater", "sweatshirt", "hoodie", "cardigan"],
    "bottoms":    ["pant", "jean", "trouser", "short", "skirt", "legging"],
    "vestidos":   ["dress", "gown", "jumpsuit", "romper"],
    "abrigos":    ["coat", "jacket", "blazer", "parka", "outerwear"],
    "calzado":    ["shoe", "boot", "sneaker", "sandal", "heel", "flat", "pump"],
    "bolsos":     ["bag", "purse", "backpack", "clutch", "tote"],
    "joyeria":    ["necklace", "earring", "bracelet", "ring", "jewelry"],
    "accesorios": ["scarf", "belt", "hat", "glove", "sunglasses", "watch", "tie"],
}

def classify_macro_category(cat_name: str) -> str:
    name = cat_name.lower()
    for macro, keywords in MACRO_CATEGORY_KEYWORDS.items():
        if any(kw in name for kw in keywords):
            return macro
    return "otros"

hgnn_macro_categories = [classify_macro_category(c) for c in hgnn_category_names]
print("Distribución por macro-categoría:", Counter(hgnn_macro_categories))

hgnn_cat_result = embedding_quality(hgnn_cat_emb_vis, hgnn_macro_categories, method="tsne")
print(f"Silhouette Score (macro-categoría, canal visual): {hgnn_cat_result['silhouette']:.4f}")

fig, ax = plt.subplots(figsize=(7, 6))
coords = hgnn_cat_result["coords"]
cmap = plt.get_cmap("tab10")
for i, macro in enumerate(sorted(set(hgnn_macro_categories))):
    mask = np.array(hgnn_macro_categories) == macro
    ax.scatter(coords[mask, 0], coords[mask, 1], color=cmap(i % 10), s=40, label=macro, alpha=0.8)
for i, name in enumerate(hgnn_category_names):
    ax.annotate(name, (coords[i, 0], coords[i, 1]), fontsize=5, alpha=0.6)
ax.legend(fontsize=8, loc="best")
_sil = hgnn_cat_result["silhouette"]
ax.set_title("HGNN — Espacio de categorías aprendido (canal visual)\n"
              f"Silhouette (macro-categoría) = {_sil:.3f}", fontsize=11)
ax.set_xticks([]); ax.set_yticks([])
plt.tight_layout()
plt.savefig("/mnt/user-data/outputs/fig_hgnn_category_space.pdf", bbox_inches="tight")
plt.show()


In [ ]:
# ============================================================
# 7.3 Vecinos más cercanos de categorías concretas -- lectura cualitativa directa
#
# En vez de depender de la sección 8 (errores por categoría, todavía no ejecutada en
# este punto), aquí se comprueba directamente si categorías semánticamente próximas
# (p. ej. camisetas/blusas) quedan próximas en el espacio APRENDIDO -- calculando el
# vecino más cercano real (coseno sobre el embedding ORIGINAL, no sobre la proyección
# t-SNE 2D, que no preserva distancias de forma fiable) para una muestra de categorías.
# ============================================================
from sklearn.metrics.pairwise import cosine_similarity

hgnn_cat_sim = cosine_similarity(hgnn_cat_emb_vis)   # (158, 158)

def hgnn_nearest_categories(cat_name: str, k: int = 5):
    if cat_name not in hgnn_category_to_idx:
        print(f"'{cat_name}' no está entre las categorías válidas de HGNN"); return
    idx = hgnn_category_to_idx[cat_name]
    sims = hgnn_cat_sim[idx].copy()
    sims[idx] = -np.inf   # excluir la propia categoría
    top_k = np.argsort(-sims)[:k]
    print(f"\nVecinos más próximos de '{cat_name}':")
    for rank, j in enumerate(top_k, 1):
        print(f"  {rank}. {hgnn_category_names[j]:<25} (cos_sim={sims[j]:.3f})")

# AJUSTAR esta lista a nombres reales de hgnn_category_names (impresos en la celda 7.1) --
# aquí van solo como ejemplo de qué categorías probar, sustituye por las que existan en tu split
for example_cat in hgnn_category_names[:5]:
    hgnn_nearest_categories(example_cat, k=5)


## 8. Errores sistemáticos de FITB por categoría *(los tres modelos)*

Se agregan los fallos de FITB por la categoría del ítem correcto (no por caso individual), y se cruza con (a) la frecuencia de esa categoría en el catálogo -- conecta con la discusión de "pool reducido" de SCE-Net en 7.5.3 -- y (b) categorías genéricas/ambiguas (p. ej. "top") frente a específicas.

In [ ]:
# ============================================================
# 8.0 Utilidad común -- accuracy de FITB agregada por categoría del GT
# ============================================================
def fitb_errors_by_category(score_fn, higher_is_better, max_questions=500, desc=""):
    """score_fn(query_ids, cand_ids) -> lista/array de scores, uno por candidato.
    Recorre fitb_questions (compartidas por los 3 modelos, sección 2) y agrega
    aciertos/total por la categoría del ítem CORRECTO (no por caso individual)."""
    questions = fitb_questions[:max_questions] if max_questions else fitb_questions
    stats = defaultdict(lambda: {"correct": 0, "total": 0})
    skipped = 0

    for q in tqdm(questions, desc=desc, leave=False):
        candidates = q["answers"]
        set_id = str(q["question"][0]).rsplit("_", 1)[0]
        gt_idx = next((i for i, c in enumerate(candidates) if c.rsplit("_", 1)[0] == set_id), None)
        if gt_idx is None:
            skipped += 1; continue

        cand_ids  = [sce_outfit_lookup.get(c) for c in candidates]
        query_ids = [i for i in (sce_outfit_lookup.get(str(t)) for t in q["question"]) if i is not None]
        if not query_ids or cand_ids[gt_idx] is None:
            skipped += 1; continue

        gt_category = item_to_category.get(cand_ids[gt_idx], sce_item_index.get(cand_ids[gt_idx], {}).get("category", "?"))

        try:
            scores = score_fn(query_ids, cand_ids)
        except Exception:
            skipped += 1; continue
        scores = np.asarray(scores, dtype=float)
        if not np.all(np.isfinite(scores)):
            scores = np.where(np.isfinite(scores), scores, -np.inf if higher_is_better else np.inf)

        predicted = int(np.argmax(scores)) if higher_is_better else int(np.argmin(scores))
        stats[gt_category]["total"] += 1
        if predicted == gt_idx:
            stats[gt_category]["correct"] += 1

    n_evaluated = sum(v["total"] for v in stats.values())
    print(f"  Preguntas evaluadas: {n_evaluated} / {len(questions)} (skipped={skipped})")
    return dict(stats)


# --- Wrappers de scoring por candidato, uno por modelo (reutilizan las funciones de la Sección 2) ---
def sce_fitb_scores(query_ids, cand_ids):
    scores = []
    for cid in cand_ids:
        if cid is None:
            scores.append(float("inf")); continue
        dists = [sce_get_pair_distance(cid, qid) for qid in query_ids]
        dists = [d for d in dists if d is not None]
        scores.append(np.mean(dists) if dists else float("inf"))
    return scores

def hgnn_fitb_scores(query_ids, cand_ids):
    context_items = [{"item_id": qi, "category": item_to_category.get(qi, "?")} for qi in query_ids]
    scores = []
    for cid in cand_ids:
        if cid is None or item_to_category.get(cid) is None:
            scores.append(float("-inf")); continue
        try:
            scores.append(hgnn_score_items(context_items + [{"item_id": cid, "category": item_to_category[cid]}]))
        except Exception:
            scores.append(float("-inf"))
    return scores

def re_fitb_scores(query_ids, cand_ids):
    if any(c is None for c in cand_ids):
        return [float("-inf")] * len(cand_ids)
    return re_score_fitb(query_ids, cand_ids)


In [ ]:
# ============================================================
# 8.1 Frecuencia de cada categoría en el catálogo
# ============================================================
catalog_freq = Counter(item_to_category.values())
print(f"Categorías distintas en el catálogo: {len(catalog_freq)}")
print("Las 10 más frecuentes:", catalog_freq.most_common(10))
print("Las 10 menos frecuentes:", catalog_freq.most_common()[-10:])


In [ ]:
# ============================================================
# 8.2 Accuracy de FITB por categoría, los 3 modelos
# (max_questions limitado por coste -- subir si el tiempo lo permite)
# ============================================================
MAX_FITB_ERROR_QUESTIONS = 500

sce_fitb_stats  = fitb_errors_by_category(sce_fitb_scores,  higher_is_better=False,
                                           max_questions=MAX_FITB_ERROR_QUESTIONS, desc="SCE-Net")
hgnn_fitb_stats = fitb_errors_by_category(hgnn_fitb_scores, higher_is_better=True,
                                           max_questions=MAX_FITB_ERROR_QUESTIONS, desc="HGNN")
re_fitb_stats   = fitb_errors_by_category(re_fitb_scores,   higher_is_better=True,
                                           max_questions=MAX_FITB_ERROR_QUESTIONS, desc="RecEngine")

all_stats = {"SCE-Net": sce_fitb_stats, "HGNN": hgnn_fitb_stats, "RecEngine": re_fitb_stats}

# Categorías presentes en los 3 (con al menos 5 preguntas evaluadas, para que el % no sea ruido)
common_cats = [c for c in catalog_freq
               if all(c in st and st[c]["total"] >= 5 for st in all_stats.values())]
common_cats.sort(key=lambda c: -catalog_freq[c])
top_cats = common_cats[:20]   # limitar a 20 para legibilidad -- la correlacion en 8.3 usa TODAS

print(f"Categorías comunes a los 3 modelos (>=5 preguntas): {len(common_cats)} -- mostrando top {len(top_cats)}")

x = np.arange(len(top_cats))
width = 0.26
fig, ax = plt.subplots(figsize=(max(10, len(top_cats) * 0.55), 5))
for i, (name, stats) in enumerate(all_stats.items()):
    accs = [stats[c]["correct"] / stats[c]["total"] for c in top_cats]
    ax.bar(x + (i - 1) * width, accs, width, label=name, color=MODEL_COLORS.get(name))
ax.set_xticks(x); ax.set_xticklabels(top_cats, rotation=60, ha="right", fontsize=8)
ax.set_ylabel("Accuracy FITB"); ax.set_ylim(0, 1)
ax.set_title(f"Accuracy de FITB por categoría del ítem correcto\n"
             f"(ordenadas por frecuencia en catálogo, de más a menos frecuente)")
ax.legend()
plt.tight_layout()
plt.savefig("/mnt/user-data/outputs/fig_fitb_errors_by_category.pdf", bbox_inches="tight")
plt.show()


In [ ]:
# ============================================================
# 8.3 Correlación (Spearman) entre frecuencia de categoría y accuracy, por modelo
# ============================================================
from scipy.stats import spearmanr

print(f"{'Modelo':<12} {'Spearman rho':>13} {'p-valor':>10}   (frecuencia catálogo vs. accuracy FITB)")
for name, stats in all_stats.items():
    cats_eval = [c for c in common_cats]   # mismo conjunto de categorías para los 3, comparabilidad
    freqs = [catalog_freq[c] for c in cats_eval]
    accs  = [stats[c]["correct"] / stats[c]["total"] for c in cats_eval]
    rho, p = spearmanr(freqs, accs)
    print(f"{name:<12} {rho:>13.3f} {p:>10.4f}")

print("\nrho > 0 y significativo (p<0.05) -> apoya la hipótesis de \'pool reducido\': "
      "categorías con menos ítems en el catálogo (menos variedad de negativos posibles) "
      "tienden a tener menor accuracy de FITB.")


## 9. Triángulos rotos — evidencia empírica de no transitividad *(los tres modelos)*

Conecta directamente con el Marco Teórico, §4.1.2, y con RQ1 (la interpretación final del hallazgo se remite a Conclusiones; aquí solo se presenta y documenta el caso). Protocolo: buscar en `compatibility_test.txt` / outfits de test tríos A, B, C donde A–B y B–C tengan score alto según el modelo, y comprobar el score A–C con el mismo scorer nativo de cada arquitectura.

In [ ]:
# ============================================================
# 9.0 Utilidad común -- búsqueda de triángulos rotos
#
# Protocolo: fijamos un item "conector" B, buscamos entre una muestra de candidatos
# los top_k que el modelo considera MÁS compatibles con B (por separado, cada uno vía
# un scorer de PAR), y comprobamos la compatibilidad ENTRE esos top_k candidatos entre
# sí. Si A y C son ambos altamente compatibles con B pero A-C sale mal puntuado, es una
# violación de transitividad -- justo el patrón de "triángulos rotos" de tu Marco Teórico
# (§4.1.2). Buscar sobre TODOS los pares sería O(n^2) e inviable; este protocolo es O(n)
# para encontrar los candidatos de B + O(top_k^2) para comprobar el triángulo, mucho más
# tratable con una muestra de cientos de items.
#
# Los 3 wrappers de abajo devuelven la compatibilidad en la MISMA dirección semántica
# (mayor = más compatible), aunque SCE-Net puntúe internamente con una distancia.
# ============================================================
def sce_pair_compat(id_a, id_b):
    d = sce_get_pair_distance(id_a, id_b)
    return -d if d is not None else None

def hgnn_pair_compat(id_a, id_b):
    cat_a, cat_b = item_to_category.get(id_a), item_to_category.get(id_b)
    if cat_a is None or cat_b is None:
        return None
    try:
        return hgnn_score_items([{"item_id": id_a, "category": cat_a}, {"item_id": id_b, "category": cat_b}])
    except Exception:
        return None

def re_pair_compat(id_a, id_b):
    return re_score_ce([id_a, id_b])


def find_broken_triangles(score_pair_fn, b_item, candidate_pool, top_k=8):
    """Devuelve una lista de (A, C, score_AB, score_BC, score_AC) ordenada de MÁS a MENOS
    'rota' (score_AC más bajo primero), entre los top_k candidatos más compatibles con B."""
    scored_with_b = []
    for item in candidate_pool:
        if item == b_item:
            continue
        s = score_pair_fn(b_item, item)
        if s is not None and np.isfinite(s):
            scored_with_b.append((item, s))
    scored_with_b.sort(key=lambda x: -x[1])
    top_matches = scored_with_b[:top_k]

    triangles = []
    for i in range(len(top_matches)):
        for j in range(i + 1, len(top_matches)):
            A, sA = top_matches[i]
            C, sC = top_matches[j]
            sAC = score_pair_fn(A, C)
            if sAC is not None and np.isfinite(sAC):
                triangles.append((A, C, sA, sC, sAC))

    triangles.sort(key=lambda t: t[4])   # score A-C ascendente -> el mas "roto" primero
    return triangles


In [ ]:
# ============================================================
# 9.1 Búsqueda de candidatos por modelo
# ============================================================
TRIANGLE_CANDIDATE_POOL_SIZE = 200   # subir si el tiempo lo permite -- O(n) en scoring contra B
TRIANGLE_TOP_K = 8                   # O(top_k^2) comprobaciones A-C

_rng2 = random.Random(1)
triangle_pool = _rng2.sample(sorted(common_ids), min(TRIANGLE_CANDIDATE_POOL_SIZE, len(common_ids)))
b_item = triangle_pool[0]   # item "conector" -- sustituye por uno concreto si quieres elegirlo a mano
print(f"Item conector B = {b_item}  (categoría: {item_to_category.get(b_item, '?')})")

sce_triangles = find_broken_triangles(sce_pair_compat, b_item, triangle_pool, top_k=TRIANGLE_TOP_K)
hgnn_triangles = find_broken_triangles(hgnn_pair_compat, b_item, triangle_pool, top_k=TRIANGLE_TOP_K)
re_triangles = find_broken_triangles(re_pair_compat, b_item, triangle_pool, top_k=TRIANGLE_TOP_K)

for name, triangles in [("SCE-Net", sce_triangles), ("HGNN", hgnn_triangles), ("RecEngine", re_triangles)]:
    if not triangles:
        print(f"[{name}] no se encontraron triángulos evaluables"); continue
    A, C, sA, sC, sAC = triangles[0]
    print(f"[{name}] candidato más 'roto': A={A} (s_AB={sA:.3f}), C={C} (s_BC={sC:.3f}), s_AC={sAC:.3f}")


In [ ]:
# ============================================================
# 9.2 Visualización del caso más claro por modelo
# ============================================================
def plot_triangle(root, item_index, b_item, A, C, sA, sC, sAC, title):
    fig, axes = plt.subplots(1, 3, figsize=(9, 3.5))
    for ax, item_id, role in zip(axes, [A, b_item, C], ["A", "B (conector)", "C"]):
        meta = item_index.get(str(item_id), {})
        ax.imshow(_load_img(root, item_index, item_id))
        ax.set_title(f"{role}\n{meta.get('category', '?')}", fontsize=9)
        ax.axis("off")
    fig.suptitle(f"{title}\ns(A,B)={sA:.3f}   s(B,C)={sC:.3f}   s(A,C)={sAC:.3f}  <- ¿triángulo roto?",
                 fontsize=11)
    plt.tight_layout()
    plt.show()

for name, triangles, root, idx in [
    ("SCE-Net",   sce_triangles,  SCE_DATA_ROOT,      sce_item_index),
    ("HGNN",      hgnn_triangles, HGNN_POLYVORE_PATH, hgnn_item_index),
    ("RecEngine", re_triangles,   RE_POLYVORE_PATH,   re_item_index),
]:
    if triangles:
        A, C, sA, sC, sAC = triangles[0]
        plot_triangle(root, idx, b_item, A, C, sA, sC, sAC, title=f"{name} — Triángulo más roto")


In [ ]:
# ============================================================
# 9.3 Tabla resumen -- ¿con qué frecuencia se rompe la transitividad?
#
# Reutiliza los umbrales de "compatible/incompatible" ya calculados en la Sección 3
# (mediana de los scores sobre compatibility_test.txt) -- así "incompatible" tiene el
# mismo criterio aquí que en el resto del notebook, en vez de inventar uno nuevo.
# Repite la búsqueda de 9.1 con varios items conectores B distintos, para que el % no
# dependa de haber tenido suerte (o mala suerte) con un único B.
# ============================================================
N_HUB_ITEMS = 15   # nº de items B distintos a probar -- cada uno cuesta ~top_k*(pool + top_k) llamadas

hub_items = _rng2.sample(sorted(common_ids), min(N_HUB_ITEMS, len(common_ids)))

def broken_fraction(score_pair_fn, incompat_threshold, hub_items, pool, top_k):
    broken, total = 0, 0
    for b in tqdm(hub_items, desc="Barriendo items conector"):
        triangles = find_broken_triangles(score_pair_fn, b, pool, top_k=top_k)
        for A, C, sA, sC, sAC in triangles:
            total += 1
            if sAC < incompat_threshold:   # los 3 wrappers de 9.0 usan "mayor = mas compatible"
                broken += 1
    return broken, total

summary_rows = []
for name, score_fn, threshold in [
    ("SCE-Net",   sce_pair_compat,  -sce_compat_threshold),   # sce_pair_compat = -distancia
    ("HGNN",      hgnn_pair_compat, hgnn_compat_threshold),
    ("RecEngine", re_pair_compat,   re_compat_threshold),
]:
    broken, total = broken_fraction(score_fn, threshold, hub_items, triangle_pool, TRIANGLE_TOP_K)
    pct = 100 * broken / total if total else float("nan")
    summary_rows.append((name, broken, total, pct))
    print(f"{name:<12} {broken:>4} / {total:<4} tríos con A-C por debajo del umbral de compatibilidad ({pct:.1f}%)")

print("\nUn % alto aquí es evidencia cuantitativa de apoyo a la discusión cualitativa sobre no "
      "transitividad (Marco Teórico, §4.1.2) -- la interpretación final de qué implica esto para "
      "RQ1 se remite al capítulo de Conclusiones, como quedó acordado.")
